In [21]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

assert torch.cuda.is_available()

CUDA: True
GPU: Tesla T4


In [22]:
from pathlib import Path
import shutil
import pandas as pd

ASSET_ROOT = Path("/kaggle/input/datasets/inhtnphng/boneage-exp006-assets/boneage-exp006-assets")
RESULT_ROOT = Path("/kaggle/input/datasets/inhtnphng/exp006-p7-tta-results/exp006-p7-tta-results")

FRIEND_REPO = ASSET_ROOT / "friend_repo"
DATA_ROOT = ASSET_ROOT / "data_goc"

WORK_ROOT = Path("/kaggle/working/exp006_p7_tta")
RUNS_ROOT = WORK_ROOT / "runs"
OUTPUT_ROOT = WORK_ROOT / "results"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

assert FRIEND_REPO.exists()
assert DATA_ROOT.exists()

manifest_src = next(
    ASSET_ROOT.rglob("development_manifest_14036.csv")
)

oof_src = next(
    RESULT_ROOT.rglob("oof_predictions.csv")
)

print("Manifest:", manifest_src)
print("OOF:", oof_src)

Manifest: /kaggle/input/datasets/inhtnphng/boneage-exp006-assets/boneage-exp006-assets/friend_repo/p0_audit/outputs/development_manifest_14036.csv
OOF: /kaggle/input/datasets/inhtnphng/exp006-p7-tta-results/exp006-p7-tta-results/oof/oof_predictions.csv


In [23]:
image_map = {
    p.stem: str(p)
    for p in DATA_ROOT.rglob("*.png")
}

print("Số ảnh:", len(image_map))

manifest = pd.read_csv(manifest_src, dtype={"image_id": str})
manifest["image_id"] = (
    manifest["image_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

manifest["image_path"] = manifest["image_id"].map(image_map)

missing = manifest["image_path"].isna().sum()
print("Ảnh thiếu:", missing)

assert len(manifest) == 14036
assert missing == 0

manifest_kaggle = WORK_ROOT / "development_manifest_kaggle.csv"
manifest.to_csv(manifest_kaggle, index=False)

oof_dst = WORK_ROOT / "oof_predictions.csv"
shutil.copy2(oof_src, oof_dst)

print("Manifest Kaggle:", manifest_kaggle)
print("OOF:", oof_dst)

Số ảnh: 14036
Ảnh thiếu: 0
Manifest Kaggle: /kaggle/working/exp006_p7_tta/development_manifest_kaggle.csv
OOF: /kaggle/working/exp006_p7_tta/oof_predictions.csv


In [25]:
for fold in range(1, 6):
    candidates = list(
        RESULT_ROOT.rglob(
            f"EXP006_P7_CONTROL_FOLD_{fold}"
        )
    )

    source = next(
        p for p in candidates
        if (p / "best_mae.ckpt").exists()
        and (p / "config_resolved.yaml").exists()
    )

    target = RUNS_ROOT / f"EXP006_P7_CONTROL_FOLD_{fold}"
    target.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source / "best_mae.ckpt", target / "best_mae.ckpt")
    shutil.copy2(
        source / "config_resolved.yaml",
        target / "config_resolved.yaml",
    )

    print("Fold", fold, "OK")

Fold 1 OK
Fold 2 OK
Fold 3 OK
Fold 4 OK
Fold 5 OK


In [26]:
assert len(list(RUNS_ROOT.glob("EXP006_P7_CONTROL_FOLD_*/best_mae.ckpt"))) == 5
assert len(pd.read_csv(oof_dst)) == 14036

In [ ]:
import sys
import subprocess

tta_script = next(
    ASSET_ROOT.rglob("evaluate_exp006_tta_oof.py")
)

cmd = [
    sys.executable,
    str(tta_script),

    "--friend-repo",
    str(FRIEND_REPO),

    "--development-manifest",
    str(manifest_kaggle),

    "--oof-csv",
    str(oof_dst),

    "--runs-root",
    str(RUNS_ROOT),

    "--run-prefix",
    "EXP006_P7_CONTROL_FOLD_",

    "--output-dir",
    str(OUTPUT_ROOT),

    "--device",
    "cuda",

    "--amp",

    "--batch-size",
    "16",

    "--num-workers",
    "2",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json

report_path = OUTPUT_ROOT / "tta_oof_report.json"
report = json.loads(report_path.read_text())

print(json.dumps(report["metrics"], indent=2))